In [ ]:
# MATLAB section 1/7 for ExplicitStimulusWhiskerData: EXPLICIT STIMULUS EXAMPLE - WHISKER STIMULATION/THALAMIC NEURON

# % EXPLICIT STIMULUS EXAMPLE - WHISKER STIMULATION/THALAMIC NEURON
# In the worksheet with analyze the stimulus effect and history effect on
# the firing of a thalamic neuron under a known stimulus consisting of
# whisker stimulation.
# Data from Demba Ba (demba@mit.edu)
#

# Python translation bootstrap for this helpfile.

# Topic: ExplicitStimulusWhiskerData
# Execution group: full
# Workflow family: data
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/ExplicitStimulusWhiskerData.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "ExplicitStimulusWhiskerData"
FAMILY = "data"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"ExplicitStimulusWhiskerData: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"ExplicitStimulusWhiskerData: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"ExplicitStimulusWhiskerData: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"ExplicitStimulusWhiskerData: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 10

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)


In [ ]:
# MATLAB section 2/7 for ExplicitStimulusWhiskerData: Load the data

# % Load the data
# MATLAB: close all;
# MATLAB: [~,~,explicitStimulusDir] = getPaperDataDirs();
#
# MATLAB: Direction=3; Neuron=1; Stim=2;
# MATLAB: datapath = fullfile(explicitStimulusDir,strcat('Dir', num2str(Direction)),...
# MATLAB:     strcat('Neuron', num2str(Neuron)), strcat('Stim', num2str(Stim)));
# MATLAB: data=load(fullfile(datapath,'trngdataBis.mat'));
#
# MATLAB: time=0:.001:(length(data.t)-1)*.001;
# MATLAB: stimData = data.t;
# MATLAB: spikeTimes = time(data.y==1);
#
# MATLAB: stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:     {'constant'});
#
# MATLAB: nst = nspikeTrain(spikeTimes);
# MATLAB: nspikeColl = nstColl(nst);
# MATLAB: cc = CovColl({stim,baseline});
# MATLAB: trial = Trial(nspikeColl,cc);
# MATLAB: trial.plot;
#
# MATLAB: figure;
# MATLAB: subplot(2,1,1);
# MATLAB: nst2 = nspikeTrain(spikeTimes);
# MATLAB: nst2.setMaxTime(21);nst.plot;
# MATLAB: subplot(2,1,2);
# MATLAB: stim.getSigInTimeWindow(0,21).plot;
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=2, matlab_line_number=30, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 2, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/7 for ExplicitStimulusWhiskerData: Fit a constant baseline and Find Stimulus Lag

# % Fit a constant baseline and Find Stimulus Lag
# We fit a constant rate (Poisson) model to the data and use the fit
# residual to determine the appropriate lag for the stimulus.
# MATLAB: clear c;
# MATLAB: selfHist = [] ; NeighborHist = []; sampleRate = 1000;
# MATLAB: c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
#
# Find Stimulus Lag (look for peaks in the cross-covariance function less
# than 1 second
# MATLAB: figure;
# MATLAB: results.Residual.xcov(stim).windowedSignal([0,1]).plot;
# MATLAB: [m,ind,ShiftTime] = max(results.Residual.xcov(stim).windowedSignal([0,1]));
# Allow for shifts of less than 1 second
# MATLAB: stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});
# MATLAB: stim = stim.shift(ShiftTime);
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:     {'constant'});
#
# MATLAB: nst = nspikeTrain(spikeTimes);
# MATLAB: nspikeColl = nstColl(nst);
# MATLAB: cc = CovColl({stim,baseline});
# MATLAB: trial = Trial(nspikeColl,cc);
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=49, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_01.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 3, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/7 for ExplicitStimulusWhiskerData: Compare constant rate model with model including stimulus effect

# % Compare constant rate model with model including stimulus effect
# Addition of the stimulus improves the fits in terms of the KS plot and
# the making the rescaled ISIs less correlated. The Point Process Residula
# also looks more "white"
# MATLAB: clear c;
# MATLAB: selfHist = [] ; NeighborHist = []; sampleRate = 1000;
# MATLAB: c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,...
# MATLAB:     NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...
# MATLAB:     sampleRate,selfHist,NeighborHist);
# MATLAB: c{2}.setName('Baseline+Stimulus');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
# MATLAB: results.plotResults;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/7 for ExplicitStimulusWhiskerData: History Effect

# % History Effect
# Determine the best history effect model using AIC, BIC, and KS statistic
# MATLAB: sampleRate=1000;
# MATLAB: delta=1/sampleRate*1;
# MATLAB: maxWindow=1; numWindows=30;
# MATLAB: windowTimes =unique(round([0 logspace(log10(delta),...
# MATLAB:     log10(maxWindow),numWindows)]*sampleRate)./sampleRate);
# MATLAB: results =Analysis.computeHistLagForAll(trial,windowTimes,...
# MATLAB:     {{'Baseline','constant'},{'Stimulus','stim'}},'BNLRCG',0,sampleRate,0);
#
# MATLAB: KSind = find(results{1}.KSStats.ks_stat == min(results{1}.KSStats.ks_stat));
# MATLAB: AICind = find((results{1}.AIC(2:end)-results{1}.AIC(1))== ...
# MATLAB:                min(results{1}.AIC(2:end)-results{1}.AIC(1)));
# MATLAB: BICind = find((results{1}.BIC(2:end)-results{1}.BIC(1))== ...
# MATLAB:                min(results{1}.BIC(2:end)-results{1}.BIC(1)));
# MATLAB: if(AICind==1)
# MATLAB:     AICind=inf;
# MATLAB: end
# MATLAB: if(BICind==1)
# MATLAB:     BICind=inf; %sometime BIC is non-decreasing and the index would be 1
# MATLAB: end
# MATLAB: windowIndex = min([KSind,AICind,BICind]) %use the minimum order model
# MATLAB: Summary = FitResSummary(results);
# MATLAB: Summary.plotSummary;
#
#
# MATLAB: clear c;
# MATLAB: if(windowIndex>1)
# MATLAB:     selfHist = windowTimes(1:windowIndex);
# MATLAB: else
# MATLAB:     selfHist = [];
# MATLAB: end
# MATLAB: NeighborHist = []; sampleRate = 1000;

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/7 for ExplicitStimulusWhiskerData: Section

# %
# MATLAB: figure;
# MATLAB: x=1:length(windowTimes);
# MATLAB: subplot(3,1,1); plot(x,results{1}.KSStats.ks_stat,'.'); axis tight; hold on;
# MATLAB: plot(x(windowIndex),results{1}.KSStats.ks_stat(windowIndex),'r*');
#
# MATLAB:  set(gca,'xtick',[]);
# MATLAB: ylabel('KS Statistic');
# MATLAB: dAIC = results{1}.AIC-results{1}.AIC(1);
# MATLAB: subplot(3,1,2); plot(x,dAIC,'.');
# MATLAB:  set(gca,'xtick',[]);
# MATLAB: ylabel('\Delta AIC');axis tight; hold on;
# MATLAB: plot(x(windowIndex),dAIC(windowIndex),'r*');
# MATLAB: dBIC = results{1}.BIC-results{1}.BIC(1);
# MATLAB: subplot(3,1,3); plot(x,dBIC,'.');
# MATLAB: ylabel('\Delta BIC'); axis tight; hold on;
# MATLAB: plot(x(windowIndex),dBIC(windowIndex),'r*');
#
# MATLAB: for i=2:length(x)
# MATLAB:    histLabels{i} = ['[' num2str(windowTimes(i-1),3) ',' num2str(windowTimes(i),3) ,']'];
# MATLAB: end
#
# MATLAB: figure;
# MATLAB: plot(x,dBIC,'.');
# MATLAB: xticks = 1:(length(histLabels));
# MATLAB: set(gca,'xtick',xticks,'xtickLabel',histLabels,'FontSize',6);
# MATLAB: if(max(xticks)>=1)
# MATLAB:     xticklabel_rotate([],90,[],'Fontsize',8);
# MATLAB: end
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=113, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_02.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 6, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=134, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_03.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 6, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 6

_ = section_index


In [ ]:
# MATLAB section 7/7 for ExplicitStimulusWhiskerData: Compare Baseline, Baseline+Stimulus Model, Baseline+History+Stimulus

# % Compare Baseline, Baseline+Stimulus Model, Baseline+History+Stimulus
# Addition of the history effect yields a model that falls within the 95%
# CI of the KS plot.
#
# MATLAB: c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,[],NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...
# MATLAB:                     sampleRate,[],[]);
# MATLAB: c{2}.setName('Baseline+Stimulus');
# MATLAB: c{3} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...
# MATLAB:                     sampleRate,windowTimes(1:windowIndex),[]);
# MATLAB: c{3}.setName('Baseline+Stimulus+Hist');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
# MATLAB: results.plotResults;
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: <synthetic MATLAB figure event #5 for ExplicitStimulusWhiskerData>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=143, matlab_snippet="<synthetic MATLAB figure event #5 for ExplicitStimulusWhiskerData>")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_04.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 7, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #6 for ExplicitStimulusWhiskerData>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=143, matlab_snippet="<synthetic MATLAB figure event #6 for ExplicitStimulusWhiskerData>")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_05.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=6 + 7, title=f"{TOPIC} Figure 006")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #7 for ExplicitStimulusWhiskerData>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=143, matlab_snippet="<synthetic MATLAB figure event #7 for ExplicitStimulusWhiskerData>")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_06.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=7 + 7, title=f"{TOPIC} Figure 007")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #8 for ExplicitStimulusWhiskerData>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=143, matlab_snippet="<synthetic MATLAB figure event #8 for ExplicitStimulusWhiskerData>")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_07.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=8 + 7, title=f"{TOPIC} Figure 008")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #9 for ExplicitStimulusWhiskerData>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=143, matlab_snippet="<synthetic MATLAB figure event #9 for ExplicitStimulusWhiskerData>")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_08.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=9 + 7, title=f"{TOPIC} Figure 009")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #10 for ExplicitStimulusWhiskerData>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=143, matlab_snippet="<synthetic MATLAB figure event #10 for ExplicitStimulusWhiskerData>")
ref_image = (MATLAB_HELP_ROOT / "ExplicitStimulusWhiskerData_09.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=10 + 7, title=f"{TOPIC} Figure 010")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "[~,~,explicitStimulusDir] = getPaperDataDirs();",
    "Direction=3; Neuron=1; Stim=2;",
    "datapath = fullfile(explicitStimulusDir,strcat('Dir', num2str(Direction)),...",
    "strcat('Neuron', num2str(Neuron)), strcat('Stim', num2str(Stim)));",
    "data=load(fullfile(datapath,'trngdataBis.mat'));",
    "time=0:.001:(length(data.t)-1)*.001;",
    "stimData = data.t;",
    "spikeTimes = time(data.y==1);",
    "stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...",
    "{'constant'});",
    "nst = nspikeTrain(spikeTimes);",
    "nspikeColl = nstColl(nst);",
    "cc = CovColl({stim,baseline});",
    "trial = Trial(nspikeColl,cc);",
    "trial.plot;",
    "figure;",
    "subplot(2,1,1);",
    "nst2 = nspikeTrain(spikeTimes);",
    "nst2.setMaxTime(21);nst.plot;",
    "subplot(2,1,2);",
    "stim.getSigInTimeWindow(0,21).plot;",
    "clear c;",
    "selfHist = [] ; NeighborHist = []; sampleRate = 1000;",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,NeighborHist);",
    "c{1}.setName('Baseline');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "figure;",
    "results.Residual.xcov(stim).windowedSignal([0,1]).plot;",
    "[m,ind,ShiftTime] = max(results.Residual.xcov(stim).windowedSignal([0,1]));",
    "stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});",
    "stim = stim.shift(ShiftTime);",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...",
    "{'constant'});",
    "nst = nspikeTrain(spikeTimes);",
    "nspikeColl = nstColl(nst);",
    "cc = CovColl({stim,baseline});",
    "trial = Trial(nspikeColl,cc);",
    "clear c;",
    "selfHist = [] ; NeighborHist = []; sampleRate = 1000;",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,...",
    "NeighborHist);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,selfHist,NeighborHist);",
    "c{2}.setName('Baseline+Stimulus');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results.plotResults;",
    "sampleRate=1000;",
    "delta=1/sampleRate*1;",
    "maxWindow=1; numWindows=30;",
    "windowTimes =unique(round([0 logspace(log10(delta),...",
    "log10(maxWindow),numWindows)]*sampleRate)./sampleRate);",
    "results =Analysis.computeHistLagForAll(trial,windowTimes,...",
    "{{'Baseline','constant'},{'Stimulus','stim'}},'BNLRCG',0,sampleRate,0);",
    "KSind = find(results{1}.KSStats.ks_stat == min(results{1}.KSStats.ks_stat));",
    "AICind = find((results{1}.AIC(2:end)-results{1}.AIC(1))== ...",
    "min(results{1}.AIC(2:end)-results{1}.AIC(1)));",
    "BICind = find((results{1}.BIC(2:end)-results{1}.BIC(1))== ...",
    "min(results{1}.BIC(2:end)-results{1}.BIC(1)));",
    "if(AICind==1)",
    "AICind=inf;",
    "end",
    "if(BICind==1)",
    "BICind=inf; %sometime BIC is non-decreasing and the index would be 1",
    "end",
    "windowIndex = min([KSind,AICind,BICind]) %use the minimum order model",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;",
    "clear c;",
    "if(windowIndex>1)",
    "selfHist = windowTimes(1:windowIndex);",
    "else",
    "selfHist = [];",
    "end",
    "NeighborHist = []; sampleRate = 1000;",
    "figure;",
    "x=1:length(windowTimes);",
    "subplot(3,1,1); plot(x,results{1}.KSStats.ks_stat,'.'); axis tight; hold on;",
    "plot(x(windowIndex),results{1}.KSStats.ks_stat(windowIndex),'r*');",
    "set(gca,'xtick',[]);",
    "ylabel('KS Statistic');",
    "dAIC = results{1}.AIC-results{1}.AIC(1);",
    "subplot(3,1,2); plot(x,dAIC,'.');",
    "set(gca,'xtick',[]);",
    "ylabel('\\Delta AIC');axis tight; hold on;",
    "plot(x(windowIndex),dAIC(windowIndex),'r*');",
    "dBIC = results{1}.BIC-results{1}.BIC(1);",
    "subplot(3,1,3); plot(x,dBIC,'.');",
    "ylabel('\\Delta BIC'); axis tight; hold on;",
    "plot(x(windowIndex),dBIC(windowIndex),'r*');",
    "for i=2:length(x)",
    "histLabels{i} = ['[' num2str(windowTimes(i-1),3) ',' num2str(windowTimes(i),3) ,']'];",
    "end",
    "figure;",
    "plot(x,dBIC,'.');",
    "xticks = 1:(length(histLabels));",
    "set(gca,'xtick',xticks,'xtickLabel',histLabels,'FontSize',6);",
    "if(max(xticks)>=1)",
    "xticklabel_rotate([],90,[],'Fontsize',8);",
    "end",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,[],NeighborHist);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,[],[]);",
    "c{2}.setName('Baseline+Stimulus');",
    "c{3} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,windowTimes(1:windowIndex),[]);",
    "c{3}.setName('Baseline+Stimulus+Hist');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "results.plotResults;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for ExplicitStimulusWhiskerData.")

# ExplicitStimulusWhiskerData: stimulus-locked spiking with binomial GLM fit.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/ExplicitStimulusWhiskerData_gold.mat"
m = loadmat(str(fixture_path))
time = np.asarray(m["time_ws"], dtype=float).reshape(-1); stimulus = np.asarray(m["stimulus_ws"], dtype=float).reshape(-1); spike = np.asarray(m["spike_ws"], dtype=float).reshape(-1)
expected_prob = np.asarray(m["expected_prob_ws"], dtype=float).reshape(-1); expected_rmse = float(np.asarray(m["expected_rmse_ws"], dtype=float).reshape(-1)[0])
fit = Analysis.fit_glm(X=stimulus[:, None], y=spike, fit_type="binomial", dt=1.0); pred_prob = np.asarray(fit.predict(stimulus[:, None]), dtype=float).reshape(-1)
window = np.ones(25, dtype=float) / 25.0; spike_prob = np.convolve(spike, window, mode="same")

fig, axes = plt.subplots(3, 1, figsize=(9.5, 7.2), sharex=False)
axes[0].plot(time, stimulus, color="k", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: explicit stimulus")
axes[0].set_ylabel("z-score")

axes[1].vlines(time[spike > 0.0], 0.6, 1.4, linewidth=0.4)
axes[1].set_ylabel("trial #1")
axes[1].set_title("Spike raster (MATLAB fixture trial)")

axes[2].plot(time, spike_prob, color="tab:blue", linewidth=1.0, label="smoothed observed")
axes[2].plot(time, pred_prob, color="tab:red", linewidth=1.0, label="python fit")
axes[2].plot(time, expected_prob, color="tab:green", linewidth=0.9, linestyle="--", label="matlab gold")
axes[2].set_title("Observed and fitted spike probability")
axes[2].set_xlabel("time [s]")
axes[2].set_ylabel("p(spike)")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

fit_rmse = float(np.sqrt(np.mean((pred_prob - spike) ** 2))); prob_max_abs = float(np.max(np.abs(pred_prob - expected_prob)))
assert pred_prob.shape == expected_prob.shape
assert prob_max_abs < 0.1
assert abs(fit_rmse - expected_rmse) < 0.1
CHECKPOINT_METRICS = {
    "prob_max_abs": float(prob_max_abs),
    "fit_rmse": float(fit_rmse),
}
CHECKPOINT_LIMITS = {
    "prob_max_abs": (0.0, 0.1),
    "fit_rmse": (0.0, 0.5),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
